# Enhanced Two-Pass RAG with Weighted Metadata & Q&A Indexing

This notebook demonstrates a **Two-Pass Retrieval-Augmented Generation (RAG)** pipeline that combines:
1. **Structured Document Ingestion**: Uses `Qwen2.5-7B-Instruct` with `outlines` to extract controlled-vocabulary metadata tags (Functional Areas & Topics with relevance weights 1–10), summaries, and synthetic Q&A pairs.
2. **Pass 1 — Hybrid Metadata Routing & Hard Constraint Filtering**: Uses deterministic regex rules paired with semantic embedding fallbacks (`ChromaDB`) to score candidate documents and disqualify documents violating numeric constraints (e.g., budget or revenue thresholds).
3. **Pass 2 — Sub-Question Vector Matching**: Performs dense cosine similarity search against synthetic Q&A pairs *only* within the filtered candidate documents.
4. **Evaluation Benchmark**: Compares Recall@2, Precision@2, and query latency against a standard Vanilla RAG baseline.

## 1. Environment Setup & Dependencies
Install required libraries for 4-bit quantized LLM inference, structured JSON generation (`outlines` + `pydantic`), and in-memory vector storage (`chromadb`).

In [1]:
!pip install -q torch transformers accelerate bitsandbytes outlines pydantic chromadb pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ibis-framework 9.5.0 r

## 2. Model Initialization, Vector Collections & Pydantic Schemas
Load `Qwen2.5-7B-Instruct` in 4-bit quantization for document metadata extraction and final answer generation, configure in-memory `ChromaDB` collections, and define strict Pydantic schemas.

In [2]:
import re
import time
from typing import Literal
import torch
import pandas as pd
import chromadb
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import outlines

# ---------------------------------------------------------------------------
# 1. Load 4-Bit Quantized Open-Weights LLM (Qwen 2.5 7B) for Colab T4 GPU
# ---------------------------------------------------------------------------
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# Wrap the HF model with Outlines to guarantee schema-compliant JSON outputs
structured_model = outlines.from_transformers(hf_model, tokenizer)

# ---------------------------------------------------------------------------
# 2. Initialize In-Memory Vector Store (ChromaDB) & Metadata Store
# ---------------------------------------------------------------------------
chroma_client = chromadb.Client()
question_collection = chroma_client.get_or_create_collection(name="doc_questions")
fa_collection = chroma_client.get_or_create_collection(name="fa_vocab")
topic_collection = chroma_client.get_or_create_collection(name="topic_vocab")

# Simulated Redis key-value store for fast document metadata lookup
redis_metadata_store: dict[str, dict] = {}

# ---------------------------------------------------------------------------
# 3. Controlled Vocabulary & Semantic Descriptions
# ---------------------------------------------------------------------------
FA_DESCRIPTIONS = {
    "IT": "IT information technology security compliance software",
    "CRM": "CRM customer relationship management workflows",
    "Finance": "Finance accounting financial management",
    "Budget": "Budget pricing cost spending software budget",
    "Marketing": "Marketing outreach campaigns SDR lead generation",
    "Sales": "Sales account executives qualification playbooks",
    "Media and Telecom": "Media and Telecom telecommunications broadcasting industry",
    "Medium Sized Business": "Medium sized business mid-market companies",
    "Enterprise": "Enterprise large organizations high-value accounts"
}

TOPIC_DESCRIPTIONS = {
    "crm": "CRM customer relationship management software",
    "monthly_budget_gte_1000": "monthly budget greater than or equal to $1,000 per month",
    "monthly_budget_lt_1000": "monthly budget less than $1,000 per month under $100",
    "annual_revenue_gt_1m": "annual revenue greater than $1 million ARR",
    "annual_revenue_lt_1m": "annual revenue less than $1 million small business under $500k",
    "media": "media companies broadcasting publishing",
    "telecom": "telecom telecommunications carriers",
    "healthcare": "healthcare medical hospitals",
    "weekly_meetings": "weekly meetings recurring discovery cadence alignment calls",
    "gap_analysis": "gap analysis needs assessment workflow mapping"
}

# Embed controlled vocabulary into ChromaDB for semantic fallback routing
fa_collection.upsert(
    ids=list(FA_DESCRIPTIONS.keys()),
    documents=list(FA_DESCRIPTIONS.values())
)

topic_collection.upsert(
    ids=list(TOPIC_DESCRIPTIONS.keys()),
    documents=list(TOPIC_DESCRIPTIONS.values())
)

AllowedFunctionalArea = Literal[
    "IT", "CRM", "Finance", "Budget", "Marketing",
    "Sales", "Media and Telecom", "Medium Sized Business", "Enterprise"
]

AllowedTopic = Literal[
    "crm", "monthly_budget_gte_1000", "monthly_budget_lt_1000",
    "annual_revenue_gt_1m", "annual_revenue_lt_1m", "media",
    "telecom", "healthcare", "weekly_meetings", "gap_analysis"
]

NAMING_GUIDELINES = """
Strict Naming Rules:
1. Include at least 1 Functional Area from:
   ['IT', 'CRM', 'Finance', 'Budget', 'Marketing', 'Sales', 'Media and Telecom', 'Medium Sized Business', 'Enterprise']
2. Include at least 1 Topic from:
   ['crm', 'monthly_budget_gte_1000', 'monthly_budget_lt_1000', 'annual_revenue_gt_1m', 'annual_revenue_lt_1m', 'media', 'telecom', 'healthcare', 'weekly_meetings', 'gap_analysis']
3. Weights must be integers from 1 to 10 (10 = highest relevance).
"""

# ---------------------------------------------------------------------------
# 4. Compact Pydantic Schemas for Structured Extraction
# ---------------------------------------------------------------------------
class FunctionalAreaTag(BaseModel):
    n: AllowedFunctionalArea = Field(..., description="Controlled functional area name")
    w: int = Field(..., ge=1, le=10, description="Relevance weight from 1 to 10")

class TopicTag(BaseModel):
    n: AllowedTopic = Field(..., description="Controlled topic name")
    w: int = Field(..., ge=1, le=10, description="Relevance weight from 1 to 10")

class QAPair(BaseModel):
    question: str = Field(..., description="Specific question answered by the document")
    answer: str = Field(..., description="Direct answer grounded in the document")

class DocumentMetadata(BaseModel):
    summary: str = Field(..., description="Concise summary of the document")
    functional_areas: list[FunctionalAreaTag] = Field(..., min_length=1)
    topics: list[TopicTag] = Field(..., min_length=1)
    qa_pairs: list[QAPair] = Field(..., min_length=1, max_length=3)

print("Model, vector collections, and Pydantic schemas initialized successfully!")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 30.7MiB/s]


Model, vector collections, and Pydantic schemas initialized successfully!


## 3. Sample Corpus & Structured Document Ingestion
Define a sample enterprise document corpus (including numeric distractors) and extract weighted metadata tags and synthetic Q&A pairs for each document.

In [3]:
sample_documents = [
    {
        "doc_id": "doc_001",
        "text": (
            "Enterprise CRM Sales Playbook: When targeting Media and Telecom companies with "
            "annual revenues exceeding $1 million, marketing and SDR teams should qualify prospects "
            "based on a minimum software budget of $1,000/month. Initial outreach should propose a "
            "recurring weekly discovery cadence to map their current CRM workflows, demonstrate "
            "product capabilities, perform a thorough gap analysis, and define clear next steps."
        )
    },
    {
        "doc_id": "doc_002",
        "text": (
            "Small Business Retail CRM Guide: Local retail shops with under $500k in annual revenue "
            "typically prefer self-service CRM tiers under $100 per month. Avoid scheduling weekly "
            "meetings; instead, send automated onboarding emails and monthly check-ins."
        )
    },
    {
        "doc_id": "doc_003",
        "text": (
            "Telecom & Media Account Qualification Criteria: High-value accounts in the Media and "
            "Telecom sector ($1M+ ARR) require dedicated account executives. Marketing campaigns "
            "should highlight ROI for budgets above $1,000/month and offer bespoke gap analysis "
            "workshops during weekly alignment calls."
        )
    },
    {
        "doc_id": "doc_004",
        "text": (
            "IT Security Compliance Checklist: All internal software tools must undergo SOC2 Type II "
            "auditing before procurement. The IT department reviews vendor security questionnaires quarterly."
        )
    }
]


def ingest_documents(docs: list[dict]) -> None:
    """Extract weighted metadata and Q&A pairs via LLM and index into metadata/vector stores."""
    for doc in docs:
        doc_id = doc["doc_id"]
        text = doc["text"]

        prompt = (
            "<|im_start|>system\n"
            "You are a precise metadata and Q&A extraction engine. Output valid JSON matching the requested schema.<|im_end|>\n"
            "<|im_start|>user\n"
            "Analyze the following document and extract structured metadata, a concise summary, "
            "and 3 specific Question & Answer pairs answered by the text.\n"
            f"{NAMING_GUIDELINES}\n\n"
            f"Document:\n{text}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )

        # Generate schema-enforced JSON output
        raw_json = structured_model(prompt, DocumentMetadata, max_new_tokens=1024)
        extracted = DocumentMetadata.model_validate_json(raw_json)

        # Store weighted metadata, summary, and full text in simulated Redis
        redis_metadata_store[doc_id] = {
            "text": text,
            "summary": extracted.summary,
            "functional_areas": {fa.n: fa.w for fa in extracted.functional_areas},
            "topics": {t.n: t.w for t in extracted.topics},
            "qa_pairs": [qa.model_dump() for qa in extracted.qa_pairs]
        }

        # Index synthetic questions into ChromaDB linked back to parent doc_id
        for idx, qa in enumerate(extracted.qa_pairs):
            question_collection.upsert(
                ids=[f"{doc_id}_q{idx}"],
                documents=[qa.question],
                metadatas=[{"doc_id": doc_id, "answer": qa.answer}]
            )

        print(
            f"Ingested {doc_id} | "
            f"FAs: {redis_metadata_store[doc_id]['functional_areas']} | "
            f"Topics: {redis_metadata_store[doc_id]['topics']}"
        )


ingest_documents(sample_documents)

Ingested doc_001 | FAs: {'Sales': 9} | Topics: {'media': 7, 'telecom': 8, 'monthly_budget_gte_1000': 10}
Ingested doc_002 | FAs: {'Medium Sized Business': 7} | Topics: {'monthly_budget_lt_1000': 9, 'weekly_meetings': 8}
Ingested doc_003 | FAs: {'Media and Telecom': 10} | Topics: {'media': 8, 'telecom': 8, 'monthly_budget_gte_1000': 9}
Ingested doc_004 | FAs: {'IT': 10} | Topics: {'monthly_budget_lt_1000': 5}


## 4. Hybrid Query Parser & Two-Pass Retrieval Engine
- **Hybrid Parser (`parse_user_prompt_hybrid`)**: Extracts exact numeric constraints (positive & negative topics), keyword matches, and semantic fallback functional areas.
- **Two-Pass Retriever (`two_pass_retrieve`)**:
  - **Pass 1**: Scores documents by weighted dot-product of Functional Areas (2x multiplier) and Topics (1x multiplier), disqualifying documents that violate numeric constraints.
  - **Pass 2**: Runs dense cosine similarity on sub-questions strictly within the top Pass 1 candidate documents.

In [4]:
def parse_user_prompt_hybrid(user_prompt: str) -> dict:
    """Extract weighted metadata tags, negative constraints, and sub-questions from a user query."""
    q_lower = user_prompt.lower()
    fas: dict[str, int] = {}
    topics: dict[str, int] = {}
    negative_topics: set[str] = set()

    # 1. Deterministic Numeric Threshold Rules (100% precision on constraint boundaries)
    if re.search(r'(more than|over|above|exceeding|>=|\+)\s*\$?1,?000', q_lower):
        topics["monthly_budget_gte_1000"] = 10
        negative_topics.add("monthly_budget_lt_1000")
    elif re.search(r'(less than|under|below|<)\s*\$?1,?000', q_lower):
        topics["monthly_budget_lt_1000"] = 10
        negative_topics.add("monthly_budget_gte_1000")

    if re.search(r'(more than|over|above|exceeding|>=|\+)\s*\$?1\s*(m|million)', q_lower):
        topics["annual_revenue_gt_1m"] = 10
        negative_topics.add("annual_revenue_lt_1m")
    elif re.search(r'(less than|under|below|<)\s*\$?1\s*(m|million)', q_lower):
        topics["annual_revenue_lt_1m"] = 10
        negative_topics.add("annual_revenue_gt_1m")

    # 2. Domain, Industry & Workflow Keyword Rules
    if "media" in q_lower:
        topics["media"] = 9
        fas["Media and Telecom"] = 10
    if "telecom" in q_lower:
        topics["telecom"] = 9
        fas["Media and Telecom"] = 10
    if "weekly" in q_lower:
        topics["weekly_meetings"] = 9
    if "gap analysis" in q_lower:
        topics["gap_analysis"] = 9
    if "crm" in q_lower:
        topics["crm"] = 8
        fas["CRM"] = 8
    if "marketing" in q_lower:
        fas["Marketing"] = 9
    if "sales" in q_lower:
        fas["Sales"] = 9
    if "healthcare" in q_lower or "medical" in q_lower:
        topics["healthcare"] = 9
    if "security" in q_lower or "it " in q_lower:
        fas["IT"] = 9

    # 3. Semantic Embedding Fallback if fewer than 2 Functional Areas matched
    if len(fas) < 2:
        fa_res = fa_collection.query(query_texts=[user_prompt], n_results=2)
        for fa_id, dist in zip(fa_res["ids"][0], fa_res["distances"][0]):
            if fa_id not in fas and dist < 1.1:
                fas[fa_id] = max(4, round((1.5 - dist) * 8))

    # 4. Decompose multi-sentence query into sub-questions for Pass 2 matching
    sub_questions = [s.strip() for s in re.split(r'[.?!]+', user_prompt) if len(s.strip()) > 10]
    if not sub_questions:
        sub_questions = [user_prompt]

    return {
        "sub_questions": sub_questions,
        "fas": fas,
        "topics": topics,
        "negative_topics": negative_topics
    }


def two_pass_retrieve(
    user_prompt: str,
    top_k_metadata: int = 3,
    top_k_final: int = 2
) -> list[dict]:
    """Retrieve top-K documents using weighted metadata filtering followed by Q&A vector search."""
    parsed = parse_user_prompt_hybrid(user_prompt)

    # PASS 1: Weighted Metadata Scoring + Hard Negative Constraint Filtering
    doc_scores: dict[str, int] = {}
    for doc_id, meta in redis_metadata_store.items():
        # Disqualify documents that violate explicit numeric constraints
        if any(neg in meta["topics"] for neg in parsed["negative_topics"]):
            doc_scores[doc_id] = -100
            continue

        score = 0
        for fa_name, weight in parsed["fas"].items():
            if fa_name in meta["functional_areas"]:
                score += weight * meta["functional_areas"][fa_name] * 2

        for topic_name, weight in parsed["topics"].items():
            if topic_name in meta["topics"]:
                score += weight * meta["topics"][topic_name]

        doc_scores[doc_id] = score

    ranked_pass_1 = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
    candidate_doc_ids = [
        doc_id for doc_id, score in ranked_pass_1[:top_k_metadata] if score > 0
    ]

    if not candidate_doc_ids:
        return []

    # PASS 2: Dense Cosine Similarity on Synthetic Questions within Candidate Docs
    results = question_collection.query(
        query_texts=parsed["sub_questions"],
        n_results=2,
        where={"doc_id": {"$in": candidate_doc_ids}}
    )

    matched_docs: dict[str, dict] = {}
    for q_idx in range(len(parsed["sub_questions"])):
        if results["metadatas"] and results["metadatas"][q_idx]:
            for i, doc_meta in enumerate(results["metadatas"][q_idx]):
                d_id = doc_meta["doc_id"]
                distance = results["distances"][q_idx][i]
                if d_id not in matched_docs or distance < matched_docs[d_id]["distance"]:
                    matched_docs[d_id] = {
                        "doc_id": d_id,
                        "distance": distance,
                        "matched_question": results["documents"][q_idx][i],
                        "matched_answer": doc_meta["answer"]
                    }

    return sorted(matched_docs.values(), key=lambda x: x["distance"])[:top_k_final]

## 5. Curated Context Assembly & Grounded Answer Generation
Assemble the retrieved documents' summaries, matched Q&A pairs, and full texts into a structured prompt for `Qwen2.5-7B-Instruct` to generate a grounded response.

In [5]:
def generate_curated_answer(user_prompt: str) -> str:
    """Retrieve relevant documents via two-pass RAG and generate a grounded LLM answer."""
    top_matches = two_pass_retrieve(user_prompt)

    if not top_matches:
        return "No relevant documents matched the weighted criteria."

    context_blocks = []
    for match in top_matches:
        d_id = match["doc_id"]
        doc_data = redis_metadata_store[d_id]
        context_blocks.append(
            f"Document ID: {d_id}\n"
            f"Summary: {doc_data['summary']}\n"
            f"Most Relevant Q&A: Q: {match['matched_question']} | A: {match['matched_answer']}\n"
            f"Full Text: {doc_data['text']}\n"
        )

    curated_context = "\n---\n".join(context_blocks)

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert assistant. Provide a hyper-specific, actionable answer "
                "using ONLY the curated context provided."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{curated_context}\n\nUser Prompt:\n{user_prompt}"
        }
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(hf_model.device)

    with torch.no_grad():
        outputs = hf_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.2
        )

    generated_ids = outputs[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


# Test query combining multi-constraint filtering + semantic intent
test_prompt = (
    "How do marketing staff reach out to potential customers who can afford to pay more than $1000 per month for CRM software? "
    "We prefer to reach out to media and telecom clients that have more than 1 million dollars annual revenue. "
    "We prefer to have weekly meetings with potential clients to understand their CRM needs, explain our product to them, "
    "work on gap analysis and identify next steps."
)

answer = generate_curated_answer(test_prompt)
print("\n================ FINAL CURATED ANSWER ================\n")
print(answer)


================ FINAL CURATED ANSWER ================

Marketing staff can effectively engage with potential media and telecom clients by following these steps:

1. **Identify High-Value Accounts**: Focus on clients with annual revenue over $1 million, as outlined in Document ID: doc_001.

2. **Qualify Leads**: Ensure that the initial leads meet the minimum software budget requirement of $1,000 per month, as specified in both documents.

3. **Propose Weekly Discovery Cadence**: Schedule weekly alignment calls to conduct a gap analysis and understand the client's current CRM workflows. This aligns with the criteria set in Document ID: doc_003.

4. **Highlight ROI**: Emphasize the return on investment (ROI) for the CRM solution, particularly for budgets above $1,000 per month, as detailed in Document ID: doc_003.

5. **Customize Engagement**: Offer bespoke gap analysis workshops tailored to the client’s specific needs, reinforcing the personalized approach emphasized in Document ID: do

## 6. Retrieval Benchmark: Two-Pass RAG vs. Vanilla RAG
Compare retrieval **Recall@2**, **Precision@2**, and **Latency** against a standard dense vector baseline (`Vanilla RAG`) that embeds raw document text without metadata filtering.

In [6]:
# 1. Build Vanilla RAG Baseline Collection (Raw Text Chunk Indexing)
vanilla_collection = chroma_client.get_or_create_collection(name="vanilla_rag_docs")

for doc_id, meta in redis_metadata_store.items():
    vanilla_collection.upsert(
        ids=[doc_id],
        documents=[meta["text"]]
    )


def vanilla_rag_retrieve(user_prompt: str, top_k: int = 2) -> list[str]:
    """Standard single-pass dense vector retrieval over raw document text."""
    results = vanilla_collection.query(
        query_texts=[user_prompt],
        n_results=top_k
    )
    return results["ids"][0] if results["ids"] and results["ids"][0] else []


# 2. Define Evaluation Dataset with Ground-Truth Document IDs
# doc_001 and doc_003 satisfy all constraints; doc_002 is a small-business distractor
eval_dataset = [
    {
        "query": test_prompt,
        "expected_doc_ids": ["doc_001", "doc_003"]
    }
]

# 3. Run Benchmark Evaluation (Recall@2, Precision@2, and Latency)
benchmark_results = []

for item in eval_dataset:
    q = item["query"]
    expected = set(item["expected_doc_ids"])

    # Measure Vanilla RAG
    t0 = time.time()
    vanilla_ids = vanilla_rag_retrieve(q, top_k=2)
    vanilla_latency = time.time() - t0
    vanilla_hits = len(expected.intersection(set(vanilla_ids)))

    # Measure Two-Pass RAG
    t0 = time.time()
    two_pass_matches = two_pass_retrieve(q, top_k_metadata=3, top_k_final=2)
    two_pass_ids = [m["doc_id"] for m in two_pass_matches]
    two_pass_latency = time.time() - t0
    two_pass_hits = len(expected.intersection(set(two_pass_ids)))

    benchmark_results.append({
        "Query": q[:50] + "...",
        "Vanilla_Recall@2": round(vanilla_hits / len(expected), 2),
        "TwoPass_Recall@2": round(two_pass_hits / len(expected), 2),
        "Vanilla_Precision@2": round(vanilla_hits / max(len(vanilla_ids), 1), 2),
        "TwoPass_Precision@2": round(two_pass_hits / max(len(two_pass_ids), 1), 2),
        "Vanilla_Latency(s)": round(vanilla_latency, 3),
        "TwoPass_Latency(s)": round(two_pass_latency, 3),
        "Vanilla_Docs": vanilla_ids,
        "TwoPass_Docs": two_pass_ids
    })

df_results = pd.DataFrame(benchmark_results)
df_results

,Query,Vanilla_Recall@2,TwoPass_Recall@2,Vanilla_Precision@2,TwoPass_Precision@2,Vanilla_Latency(s),TwoPass_Latency(s),Vanilla_Docs,TwoPass_Docs
0,How do marketing staff reach out to potential ...,0.5,1.0,0.5,1.0,0.307,0.519,"[doc_001, doc_002]","[doc_003, doc_001]"
